# 03 — Topic Library Builder

Generate educational short-video topics from a saved knowledge tree, manually approve the strongest ideas, and add only those approved topics to the persistent SQLite topic library.

This notebook:

1. Loads a knowledge tree created by Notebook 02.
2. Reads category coverage from the existing SQLite topic library.
3. Automatically selects a category path that does not yet have enough usable topics, or uses a manual override.
4. Generates a reusable set of topic candidates with the local LLM.
5. Lets you review and approve candidates by index.
6. Adds approved topics to `data/topic_library.db`.
7. Displays the current topic-library inventory.

The live SQLite database should remain local and should not be committed to Git.


## Load the project


In [1]:
import json
import random
import sqlite3
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.prompts import load_prompt
from educational_shorts.topic_library import (
    add_topic,
    initialize_topic_library,
)
from educational_shorts.topics import (
    collect_node_paths,
    generate_topics,
)
from educational_shorts.tree import load_tree

print(f"Project root: {PROJECT_ROOT}")


Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

Automatic selection skips a category once it has at least `MIN_COVERING_TOPICS_PER_CATEGORY` topics whose status is listed in `COVERED_TOPIC_STATUSES`.

`completed` is included because Notebook 12 changes an approved topic to completed after producing its video. Failed or rejected topics do not count as coverage, so that branch can be tried again.

Set `CATEGORY_PATH_OVERRIDE` to a complete path when you intentionally want a particular branch, even if it is already covered.

Example:

```python
CATEGORY_PATH_OVERRIDE = ["Science", "Biology", "Microbiology"]
```

Leave it as `None` to automatically choose from the remaining uncovered paths.


In [2]:
ROOT_CATEGORY = "Science"
TREE_FILENAME = None

TOPIC_COUNT = 10
MIN_CATEGORY_DEPTH = 2
MAX_CATEGORY_DEPTH = 3

# A category is skipped after it has this many usable topics in SQLite.
MIN_COVERING_TOPICS_PER_CATEGORY = 1
COVERED_TOPIC_STATUSES = (
    "approved",
    "processing",
    "completed",
)

# Use an integer for repeatable selection or None for fresh randomness.
# Because covered paths are removed first, the same integer still advances
# to a different category as the database fills.
CATEGORY_SELECTION_SEED = 42
TOPIC_GENERATION_SEED = 42
TOPIC_GENERATION_TEMPERATURE = 0.7

# Example:
# CATEGORY_PATH_OVERRIDE = ["Science", "Biology", "Microbiology"]
CATEGORY_PATH_OVERRIDE = None

TREE_DIRECTORY = PROJECT_ROOT / "data" / "knowledge_tree"
TREE_PATH = TREE_DIRECTORY / (
    TREE_FILENAME
    or f"{ROOT_CATEGORY.lower().replace(' ', '_')}.json"
)

TOPIC_DATABASE_PATH = PROJECT_ROOT / "data" / "topic_library.db"

print(f"Tree path: {TREE_PATH}")
print(f"Topic database: {TOPIC_DATABASE_PATH}")


Tree path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
Topic database: c:\Users\hitch\python_files\educational_shorts\data\topic_library.db


## Load the knowledge tree and initialize the database


In [3]:
if not TREE_PATH.exists():
    raise FileNotFoundError(
        f"Knowledge tree not found at {TREE_PATH}. "
        "Run Notebook 02 before building the topic library."
    )

knowledge_tree = load_tree(TREE_PATH)
initialize_topic_library(TOPIC_DATABASE_PATH)
topic_system_prompt = load_prompt("video_topic_generation")

print(f"Loaded tree: {knowledge_tree.name}")
print("Topic library initialized.")
print("Topic-generation prompt loaded.")


Loaded tree: Science
Topic library initialized.
Topic-generation prompt loaded.


## Inspect category coverage

This cell compares every knowledge-tree path in the configured depth range with the paths already represented in SQLite.

- **Open** paths are eligible for automatic selection.
- **Covered** paths already meet the configured topic threshold and are skipped.


In [4]:
available_category_paths = collect_node_paths(
    tree=knowledge_tree,
    min_depth=MIN_CATEGORY_DEPTH,
    max_depth=MAX_CATEGORY_DEPTH,
)

if MIN_COVERING_TOPICS_PER_CATEGORY < 1:
    raise ValueError("MIN_COVERING_TOPICS_PER_CATEGORY must be at least 1.")

if not COVERED_TOPIC_STATUSES:
    raise ValueError("COVERED_TOPIC_STATUSES cannot be empty.")

status_placeholders = ", ".join("?" for _ in COVERED_TOPIC_STATUSES)

with sqlite3.connect(TOPIC_DATABASE_PATH) as connection:
    connection.row_factory = sqlite3.Row

    coverage_rows = connection.execute(
        f"""
        SELECT category_path, COUNT(*) AS topic_count
        FROM topics
        WHERE status IN ({status_placeholders})
        GROUP BY category_path
        """,
        tuple(COVERED_TOPIC_STATUSES),
    ).fetchall()

covered_topic_counts = {
    tuple(json.loads(row["category_path"])): int(row["topic_count"])
    for row in coverage_rows
}

covered_category_paths = []
uncovered_category_paths = []

for path in available_category_paths:
    topic_count = covered_topic_counts.get(tuple(path), 0)

    if topic_count >= MIN_COVERING_TOPICS_PER_CATEGORY:
        covered_category_paths.append(path)
    else:
        uncovered_category_paths.append(path)

print(f"Available category paths: {len(available_category_paths)}")
print(f"Covered category paths: {len(covered_category_paths)}")
print(f"Remaining category paths: {len(uncovered_category_paths)}")
print()

for index, path in enumerate(available_category_paths):
    topic_count = covered_topic_counts.get(tuple(path), 0)
    is_covered = topic_count >= MIN_COVERING_TOPICS_PER_CATEGORY
    label = "COVERED" if is_covered else "OPEN"

    print(
        f"[{index}] [{label}: {topic_count}] "
        f"{' > '.join(path)}"
    )


Available category paths: 72
Covered category paths: 1
Remaining category paths: 71

[0] [OPEN: 0] Science > Biology > Cell Biology
[1] [OPEN: 0] Science > Biology > Cell Biology > Cell Structure
[2] [OPEN: 0] Science > Biology > Cell Biology > Cell Membrane
[3] [OPEN: 0] Science > Biology > Cell Biology > Cytoplasmic Organelles
[4] [OPEN: 0] Science > Biology > Cell Biology > Nucleus and DNA
[5] [OPEN: 0] Science > Biology > Cell Biology > Cell Division
[6] [OPEN: 0] Science > Biology > Cell Biology > Genetic Information Flow
[7] [OPEN: 0] Science > Biology > Cell Biology > Cell Signaling
[8] [OPEN: 0] Science > Biology > Cell Biology > Cell Cycle Regulation
[9] [OPEN: 0] Science > Biology > Genetics
[10] [OPEN: 0] Science > Biology > Genetics > DNA Structure
[11] [OPEN: 0] Science > Biology > Genetics > Gene Expression
[12] [OPEN: 0] Science > Biology > Genetics > Inheritance Patterns
[13] [OPEN: 0] Science > Biology > Genetics > Mutations and Variants
[14] [OPEN: 0] Science > Biolog

## Select an uncovered category

Automatic selection only considers paths labeled `OPEN` above. A manual override is treated as intentional and may select a covered path.


In [5]:
if CATEGORY_PATH_OVERRIDE is not None:
    selected_category_path = list(CATEGORY_PATH_OVERRIDE)

    if selected_category_path not in available_category_paths:
        raise ValueError(
            "CATEGORY_PATH_OVERRIDE does not match a path in the loaded "
            "knowledge tree within the configured depth range: "
            f"{selected_category_path}"
        )

    existing_topic_count = covered_topic_counts.get(
        tuple(selected_category_path),
        0,
    )

    if existing_topic_count >= MIN_COVERING_TOPICS_PER_CATEGORY:
        print(
            "Manual override selected an already-covered category "
            f"({existing_topic_count} usable topics)."
        )
else:
    if not uncovered_category_paths:
        raise RuntimeError(
            "Every category path in the configured depth range is covered. "
            "Increase the depth range, raise MIN_COVERING_TOPICS_PER_CATEGORY, "
            "change ROOT_CATEGORY, or use CATEGORY_PATH_OVERRIDE."
        )

    category_rng = random.Random(CATEGORY_SELECTION_SEED)
    selected_category_path = list(
        category_rng.choice(uncovered_category_paths)
    )

print("Selected category:")
print(" > ".join(selected_category_path))


Selected category:
Science > Biology > Genetics > Population Genetics


## Generate topic candidates

This is the only LLM-generation step in this notebook. With automatic selection, candidates are generated only for an uncovered category. Change `TOPIC_GENERATION_SEED` when you want a different set for the same category.


In [6]:
generated_topics = generate_topics(
    category_path=selected_category_path,
    system_prompt=topic_system_prompt,
    count=TOPIC_COUNT,
    temperature=TOPIC_GENERATION_TEMPERATURE,
    seed=TOPIC_GENERATION_SEED,
)

print(f"Generated {len(generated_topics.topics)} topic candidates.")


Generated 10 topic candidates.


## Preview candidates


In [7]:
print("CATEGORY")
print(" > ".join(selected_category_path))
print()

for index, topic in enumerate(generated_topics.topics):
    print(f"[{index}] {topic.title}")
    print(f"    {topic.learning_objective}")
    print()


CATEGORY
Science > Biology > Genetics > Population Genetics

[0] How Allele Frequencies Change Over Time
    Understand how allele frequencies in a population can shift due to natural selection, mutation, and genetic drift.

[1] The Hardy-Weinberg Principle Explained
    Learn how the Hardy-Weinfeld equation helps predict genotype frequencies in a non-evolving population.

[2] Why Genetic Drift Matters in Small Populations
    Explore how random events can significantly alter gene frequencies in small, isolated populations.

[3] Gene Flow: How Populations Stay Connected Genetically
    Discover how the movement of individuals between populations affects genetic diversity and evolution.

[4] Mutation Rates and Their Impact on Evolution
    Understand how mutation rates contribute to genetic variation and influence evolutionary change over time.

[5] Natural Selection in Action: A Population Perspective
    See how natural selection acts on entire populations, not just individual organis

## Choose approved topics

Edit `APPROVED_TOPIC_INDEXES` after reviewing the candidates above.

Example:

```python
APPROVED_TOPIC_INDEXES = [0, 2, 5]
```

Leave the list empty until you are ready. This prevents accidental database inserts.


In [10]:
APPROVED_TOPIC_INDEXES = [i for i in range(10)]

if not APPROVED_TOPIC_INDEXES:
    print(
        "No topics selected yet. Add candidate indexes to "
        "APPROVED_TOPIC_INDEXES, then rerun this cell."
    )
else:
    unique_indexes = list(dict.fromkeys(APPROVED_TOPIC_INDEXES))

    for index in unique_indexes:
        if index < 0 or index >= len(generated_topics.topics):
            raise IndexError(
                f"Topic index {index} is outside the valid range "
                f"0 to {len(generated_topics.topics) - 1}."
            )

    print("Topics selected for approval:")
    print()

    for index in unique_indexes:
        topic = generated_topics.topics[index]
        print(f"[{index}] {topic.title}")


Topics selected for approval:

[0] How Allele Frequencies Change Over Time
[1] The Hardy-Weinberg Principle Explained
[2] Why Genetic Drift Matters in Small Populations
[3] Gene Flow: How Populations Stay Connected Genetically
[4] Mutation Rates and Their Impact on Evolution
[5] Natural Selection in Action: A Population Perspective
[6] Inbreeding Depression and Its Genetic Consequences
[7] The Role of Migration in Genetic Diversity
[8] Founder Effect: How Small Groups Shape Genes
[9] Why Isolation Leads to Genetic Divergence


## Add approved topics to SQLite

Run this only after setting `APPROVED_TOPIC_INDEXES`.

Duplicate titles are skipped rather than inserted twice. Once enough approved topics are added, this category will be marked covered the next time the notebook is run from the top.


In [11]:
if not APPROVED_TOPIC_INDEXES:
    raise RuntimeError(
        "APPROVED_TOPIC_INDEXES is empty. Review the candidates and select "
        "at least one index before inserting topics."
    )

unique_indexes = list(dict.fromkeys(APPROVED_TOPIC_INDEXES))
added_count = 0
duplicate_count = 0

for index in unique_indexes:
    if index < 0 or index >= len(generated_topics.topics):
        raise IndexError(
            f"Topic index {index} is outside the valid range "
            f"0 to {len(generated_topics.topics) - 1}."
        )

    topic = generated_topics.topics[index]

    topic_id = add_topic(
        database_path=TOPIC_DATABASE_PATH,
        topic=topic,
        category_path=topic.category_path,
        status="approved",
    )

    if topic_id is None:
        duplicate_count += 1
        print(f"Already exists: {topic.title}")
    else:
        added_count += 1
        print(f"Added [{topic_id}]: {topic.title}")

print()
print(f"Added: {added_count}")
print(f"Duplicates skipped: {duplicate_count}")


Added [11]: How Allele Frequencies Change Over Time
Added [12]: The Hardy-Weinberg Principle Explained
Added [13]: Why Genetic Drift Matters in Small Populations
Added [14]: Gene Flow: How Populations Stay Connected Genetically
Added [15]: Mutation Rates and Their Impact on Evolution
Added [16]: Natural Selection in Action: A Population Perspective
Added [17]: Inbreeding Depression and Its Genetic Consequences
Added [18]: The Role of Migration in Genetic Diversity
Added [19]: Founder Effect: How Small Groups Shape Genes
Added [20]: Why Isolation Leads to Genetic Divergence

Added: 10
Duplicates skipped: 0


## Review the topic-library inventory


In [12]:
with sqlite3.connect(TOPIC_DATABASE_PATH) as connection:
    connection.row_factory = sqlite3.Row

    status_rows = connection.execute(
        '''
        SELECT status, COUNT(*) AS topic_count
        FROM topics
        GROUP BY status
        ORDER BY status
        '''
    ).fetchall()

    topic_rows = connection.execute(
        '''
        SELECT
            id,
            title,
            category_path,
            status,
            created_at_utc,
            selected_at_utc,
            completed_at_utc,
            final_video_path,
            failure_message
        FROM topics
        ORDER BY id
        '''
    ).fetchall()

print("STATUS COUNTS")
for row in status_rows:
    print(f"{row['status']}: {row['topic_count']}")

print()
print("TOPICS")

for row in topic_rows:
    path = json.loads(row["category_path"])

    print(f"[{row['id']}] {row['title']}")
    print(f"    Category: {' > '.join(path)}")
    print(f"    Status: {row['status']}")

    if row["final_video_path"]:
        print(f"    Video: {row['final_video_path']}")

    if row["failure_message"]:
        print(f"    Failure: {row['failure_message']}")

    print()


STATUS COUNTS
approved: 19
completed: 1

TOPICS
[1] How Bacteria Communicate
    Category: Science > Biology > Microbiology
    Status: completed
    Video: c:\Users\hitch\python_files\educational_shorts\data\videos\bacteria_talk_with_chemicals\final.mp4

[2] How Do Bacteria Communicate?
    Category: Science > Biology > Microbiology
    Status: approved

[3] What Is a Biofilm, and Why Does It Matter?
    Category: Science > Biology > Microbiology
    Status: approved

[4] How Do Antimicrobial Resistance Genes Spread?
    Category: Science > Biology > Microbiology
    Status: approved

[5] What Are the Differences Between Viruses and Bacteria?
    Category: Science > Biology > Microbiology
    Status: approved

[6] How Do Microbes Help Break Down Plastic?
    Category: Science > Biology > Microbiology
    Status: approved

[7] What Is the Role of Endospores in Bacterial Survival?
    Category: Science > Biology > Microbiology
    Status: approved

[8] How Do Microbes Influence Human Di

## Next step

Rerun this notebook from the top whenever you want to fill another branch. Automatic selection will skip categories already represented by approved, processing, or completed topics and continue through the remaining paths.

When every eligible path is covered, the selection cell stops with a completion message instead of generating duplicates.

Notebook 12 claims one topic whose status is `approved`, marks it `processing`, runs the production pipeline, and finally marks it `completed` or `failed`.
